In [1]:
!pip install torch datasets transformers gradio peft

In [8]:
from datasets import load_dataset

# Fixed repository path
dataset = load_dataset("Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset")
print(dataset)


README.md:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

CyberSec-Dataset_escaped.jsonl: reconstructing file:   0%|          |  0.00B /  195MB            

CyberSec-Dataset_escaped.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['system', 'user', 'assistant'],
        num_rows: 53201
    })
})


In [9]:
sample = dataset['train'][0]

print("--- SYSTEM ---")
print(sample.get('system', 'No system prompt found'))

print("\n--- USER ---")
print(sample.get('user', 'No user prompt found'))

print("\n--- ASSISTANT ANSWER ---")
print(sample.get('assistant', 'No assistant answer found'))


--- SYSTEM ---
You are a highly specialized AI assistant for advanced cyber-defense whose mission is to deliver accurate, in-depth, actionable guidance on information-security principles—confidentiality, integrity, availability, authenticity, non-repudiation, and privacy—by offering concise executive summaries that drill down into technical detail, industry standards, and threat models while referencing frameworks such as NIST CSF and MITRE ATT&CK; you may share defensive scripts, detection rules, lab-safe PoC payloads, exploit snippets, and hardening checklists clearly marked for educational/testing use only, redacting or stubbing any data that could cause real harm in production. You must never generate or improve ransomware, wipers, botnets, RATs, phishing kits, social-engineering lures, or any instructions that facilitate fraud, data theft, unauthorized intrusion, or the defeat of security controls—in such cases you must briefly refuse with an apology and a one-sentence explanation

In [12]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Initialize and load the model components (Fixes NameError)
model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. Define the response generation function
def generate_security_response(user_message, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    # Format input with Qwen chat template
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    # Generate output safely on GPU
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,  # Fixed escaped asterisks (\*\*)
            max_new_tokens=512,
            temperature=0.6,
            top_p=0.9
        )

    # Trim the prompt tokens out of the generation
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# Cybersecurity Expert System Persona
default_system = (
    "You are an expert Cybersecurity Researcher and Bug Bounty Advisor. "
    "Explain security concepts clearly, analyze vulnerabilities, and write secure defensive code patches."
)

# 3. Build Web GUI
demo = gr.Interface(
    fn=generate_security_response,
    inputs=[
        gr.Textbox(lines=4, placeholder="Paste code or ask about XSS, SQLi, Buffer Overflows...", label="Bug Bounty / Security Query"),
        gr.Textbox(value=default_system, lines=2, label="System Persona")
    ],
    outputs=gr.Textbox(lines=12, label="Defensive Security Response"),
    title="🛡️ Cybersecurity & Bug Bounty AI Assistant"  # Fixed &amp; HTML literal
)

# Launch public web link
demo.launch(share=True)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://32fb2f04081f947722.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model onto the T4 GPU in 16-bit half precision
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model and Tokenizer loaded successfully on GPU!")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model and Tokenizer loaded successfully on GPU!
